# Chemprop Affinity Debugging

Small end-to-end workflow for debugging the chemprop affinity pipeline:

1. Build a tiny `affinity_split_manifest`-style dataset (10 ligands, each in train/val/test)
2. Train a chemprop model on that dataset
3. Inspect loss curves and evaluation metrics

In [ ]:
from pathlib import Path

import json
import pandas as pd
import matplotlib.pyplot as plt

from data_processing.common.constants import AFFINITY_SPLIT_MANIFEST_CSV, MVP_ROOT
from models.chemprop_affinity.train import train_chemprop_affinity

## 1. Generate a tiny test dataset

Sample 10 ligands from the full `affinity_split_manifest.csv` for a single protein,
then duplicate each ligand into `train`, `val`, and `test` so every split has the same molecules.

In [ ]:
UNIPROT_ID = "P07550"
NUM_LIGANDS = 10
SPLITS = ("train", "val", "test")

TEST_MANIFEST_PATH = MVP_ROOT / "processed" / "affinity_split_manifest_test_10ligands.csv"
CONFIG_PATH = MVP_ROOT / "configs" / "chemprop_affinity_test_10ligands.json"
RUN_DIR = MVP_ROOT / "runs" / "chemprop_ligand_test_10ligands"

In [ ]:
def build_test_affinity_split_manifest(
    source_csv: Path,
    output_csv: Path,
    uniprot_id: str,
    num_ligands: int,
    splits: tuple[str, ...] = ("train", "val", "test"),
) -> pd.DataFrame:
    """Downsample ligands and repeat each one across all splits."""
    source_df = pd.read_csv(source_csv)
    ligand_rows = (
        source_df[source_df["uniprot_id"] == uniprot_id]
        .drop_duplicates("ligand")
        .head(num_ligands)
    )

    rows = []
    for _, row in ligand_rows.iterrows():
        for split in splits:
            rows.append(
                {
                    "uniprot_id": row["uniprot_id"],
                    "protein_idx": row["protein_idx"],
                    "ligand": row["ligand"],
                    "ligand_idx": row["ligand_idx"],
                    "affinity": row["affinity"],
                    "split": split,
                }
            )

    out_df = pd.DataFrame(rows)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(output_csv, index=False)
    return out_df


test_manifest_df = build_test_affinity_split_manifest(
    source_csv=AFFINITY_SPLIT_MANIFEST_CSV,
    output_csv=TEST_MANIFEST_PATH,
    uniprot_id=UNIPROT_ID,
    num_ligands=NUM_LIGANDS,
    splits=SPLITS,
)

print(f"Wrote {len(test_manifest_df)} rows to {TEST_MANIFEST_PATH}")
print(f"Unique ligands: {test_manifest_df['ligand'].nunique()}")
print(test_manifest_df.groupby("split").size())
test_manifest_df.head(9)

## 2. Write experiment config

In [ ]:
experiment_config = {
    "experiment_name": "chemprop_ligand_test_10ligands",
    "uniprot_id": UNIPROT_ID,
    "paths": {
        "affinity_split_csv": str(TEST_MANIFEST_PATH.relative_to(MVP_ROOT)),
        "runs_dir": "runs",
    },
    "model": {
        "message_hidden_dim": 300,
        "message_depth": 3,
        "ffn_hidden_dim": 300,
        "ffn_layers": 2,
        "dropout": 0.1,
        "batchnorm": True,
    },
    "training": {
        "batch_size": 4,
        "max_epochs": 20,
        "learning_rate": 0.001,
        "seed": 0,
    },
    "overwrite_existing_run": True,
}

CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(json.dumps(experiment_config, indent=2) + "\n", encoding="utf-8")
print(f"Wrote config to {CONFIG_PATH}")
experiment_config

## 3. Train chemprop model

In [ ]:
train_chemprop_affinity(CONFIG_PATH)

## 4. Inspect results

In [ ]:
loss_history_df = pd.read_csv(RUN_DIR / "training_loss_history.csv")
loss_history_df.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history_df["epoch"], loss_history_df["train_loss"], label="train")
ax.plot(loss_history_df["epoch"], loss_history_df["val_loss"], label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("chemprop_ligand_test_10ligands train/val loss")
ax.legend()
plt.show()

In [ ]:
for split in SPLITS:
    results_path = RUN_DIR / f"results_{split}.txt"
    print(results_path.read_text())

In [ ]:
from IPython.display import Image, display

for split in SPLITS:
    scatter_path = RUN_DIR / f"scatter_{split}.png"
    print(f"{split} scatter plot")
    display(Image(filename=str(scatter_path)))